# Chatbot with Tool Calls

A chatbot to fetch papers and return information about them using an LLM.

In [53]:
import arxiv
import json
import os
from openai import OpenAI
from dotenv import load_dotenv
import pprint

In [12]:
papers_dir = 'data'
papers_path = os.path.join(papers_dir, 'papers_info.json')
os.makedirs(papers_dir, exist_ok=True)

## Tools

### Definition

In [97]:
storage_path = papers_path

def search_arxiv(query: str, max_results: int = 5) -> list[str]:
    """
    Search for papers on arXiv based on a query string and store the results in a specified path.
    
    Args:
        query (str): The search query string.
        max_results (int): The maximum number of results to return.
        
    Returns:
        List of arxiv Paper IDs for the found papers.
    """
    client = arxiv.Client()
    search = arxiv.Search(
        query=query,
        max_results=max_results,
        sort_by=arxiv.SortCriterion.Relevance
    )
    papers = client.results(search)
    print(f"===(search_arxiv)=== Found {max_results} papers for query: '{query}'")

    # load existing data
    try:
        with open(storage_path, 'r') as f:
            papers_info = json.load(f)
            print(f"===(search_arxiv)=== Loaded existing papers info from {storage_path}")
    except (FileNotFoundError, json.JSONDecodeError):
        print(f"===(search_arxiv)=== No existing data found at {storage_path}. Initializing new storage.")
        papers_info = {}
        
    paper_ids = []
    for paper in papers:
        paper_id = str.split(paper.entry_id, '/')[-1]
        paper_ids.append(paper_id)
        if paper_id not in papers_info:
            papers_info[paper_id] = {
                'title': paper.title,
                'authors': [str(author) for author in paper.authors],
                'summary': paper.summary,
                'published': paper.published.isoformat(),
                'updated': paper.updated.isoformat(),
                'pdf_url': paper.pdf_url
            }
    
    # save updated data
    with open(storage_path, 'w') as f:
        json.dump(papers_info, f, indent=2)
    print(f"===(search_arxiv)=== Stored paper information in {storage_path}")
    return paper_ids

In [98]:
search_arxiv(
    query="machine learning",
    max_results=3
)

===(search_arxiv)=== Found 3 papers for query: 'machine learning'
===(search_arxiv)=== Loaded existing papers info from data\papers_info.json
===(search_arxiv)=== Stored paper information in data\papers_info.json
===(search_arxiv)=== Stored paper information in data\papers_info.json


['2306.04338v1', '2006.16189v4', '2201.12150v2']

In [86]:
from typing import Optional

def get_paper_info(paper_id: str) -> Optional[dict]:
    """
    Retrieve information about a specific paper by its arXiv ID.
    
    Args:
        paper_id (str): The arXiv ID of the paper.
        
    Returns:
        dict: A dictionary containing the paper's information.
    """
    try:
        with open(storage_path, 'r') as f:
            papers_info = json.load(f)
    except json.JSONDecodeError:
        print(f"===(get_paper_info)=== Data at {storage_path} is corrupted or not in valid JSON format.")
    except FileNotFoundError:
        print(f"===(get_paper_info)=== No valid storage found at {storage_path}.")
        return None
        
    if paper_id in papers_info:
        return papers_info[paper_id]
    else:
        print(f"===(get_paper_info)=== Paper ID {paper_id} not found in storage.")

In [87]:
get_paper_info('2006.16189v4')

{'title': 'DOME: Recommendations for supervised machine learning validation in biology',
 'authors': ['Ian Walsh',
  'Dmytro Fishman',
  'Dario Garcia-Gasulla',
  'Tiina Titma',
  'Gianluca Pollastri',
  'The ELIXIR Machine Learning focus group',
  'Jen Harrow',
  'Fotis E. Psomopoulos',
  'Silvio C. E. Tosatto'],
 'summary': 'Modern biology frequently relies on machine learning to provide predictions and improve decision processes. There have been recent calls for more scrutiny on machine learning performance and possible limitations. Here we present a set of community-wide recommendations aiming to help establish standards of supervised machine learning validation in biology. Adopting a structured methods description for machine learning based on data, optimization, model, evaluation (DOME) will aim to help both reviewers and readers to better understand and assess the performance and limitations of a method or outcome. The recommendations are formulated as questions to anyone wishin

### Schema

In [99]:
arxiv_tools = [
    {
        "type": "function",
        "function": {
            "name": "search_arxiv",
            "description": "Search for papers on arXiv based on a query string, store the results and get paper IDs.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query string."
                    },
                    "max_results": {
                        "type": "integer",
                        "description": "The maximum number of results to return.",
                        "default": 5
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_paper_info",
            "description": "Retrieve information about a specific paper by its arXiv ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "paper_id": {
                        "type": "string",
                        "description": "The arXiv ID of the paper."
                    }
                },
                "required": ["paper_id"]
            }
        }
    }
]

### Mapping

In [100]:
arxiv_tool_map = {
    "search_arxiv": search_arxiv,
    "get_paper_info": get_paper_info
}

In [101]:
def call_tool(tool_name: str, tool_args, tool_map: dict):
    """
    Executes a tool based on its name and arguments.
    """
    try:    
        if tool_name in tool_map:
            tool_result = tool_map[tool_name](**tool_args)
            
            if tool_result is not None:
                print(f"===(call_tool)=== Tool '{tool_name}' executed successfully.")
                return tool_result
            if tool_result is None:
                print(f"===(call_tool)=== Tool '{tool_name}' executed successfully with no return value.")
                return None
        else:
            raise ValueError(f"===(call_tool)=== Tool {tool_name} not found.")
    except Exception as e:
        print(f"===(call_tool)=== Error executing tool '{tool_name}': {e}")
        raise e

## LLM Chat

In [102]:
load_dotenv(dotenv_path=".env")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
print(f"===(main)=== Loaded OPENAI_API_KEY: { '*' * 8 if OPENAI_API_KEY else None }")

llm_client = OpenAI(api_key=OPENAI_API_KEY)

===(main)=== Loaded OPENAI_API_KEY: ********


In [119]:
from IPython.display import display, HTML

def process_user_query(system_instruction: str, user_query: str, messages: list, llm_client: OpenAI, tools: list, tool_map: dict, max_turns: int = 5):
    """
    Process a user query using the LLM and available tools.
    
    Args:
        system_instruction (str): The system instruction for the LLM.
        user_query (str): The user's query.
        messages (list): The conversation history.
        llm_client (OpenAI): The OpenAI client instance.
        tools (list): The list of available tools.
        tool_map (dict): A mapping of tool names to their corresponding functions.
        max_turns (int): The maximum number of interaction turns.
    """
    messages.append({"role": "user", "content": user_query})
    
    finish = False
    turns = 0
    while not finish and turns < max_turns:
        turns += 1
        response = llm_client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=[
                {"role": "system", "content": system_instruction},
                *messages
            ],
            tools=tools,
            tool_choice="auto"
        )
        
        message = response.choices[0].message
        if message.tool_calls:
            for tool_call in message.tool_calls:
                tool_name = tool_call.function.name
                tool_args = json.loads(tool_call.function.arguments)
                print(f"===(process_user_query)=== Invoking tool: {tool_name} with args: {tool_args}")
                
                tool_result = call_tool(tool_name, tool_args, tool_map)
                
                messages.append(message.model_dump())
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": tool_name,
                    "content": str(tool_result)
                })
        else:
            messages.append({
                "role": "assistant",
                "content": message.content
            })
            print("\n\n========== RESPONSE ==========\n")
            print(message.content)
            print("==================================")
            finish = True

In [117]:
instruction = "You are an AI research assistant that helps users find and summarize academic papers from arXiv. Use the provided tools to search for papers and retrieve their information as needed."
query = "Find me 2 recent papers on reinforcement learning and give me a summary of one of them."
history = []

In [120]:
process_user_query(
    system_instruction=instruction,
    user_query=query,
    messages=history,
    llm_client=llm_client,
    tools=arxiv_tools,
    tool_map=arxiv_tool_map
)

===(process_user_query)=== Invoking tool: search_arxiv with args: {'query': 'reinforcement learning', 'max_results': 2}
===(search_arxiv)=== Found 2 papers for query: 'reinforcement learning'
===(search_arxiv)=== Loaded existing papers info from data\papers_info.json
===(search_arxiv)=== Stored paper information in data\papers_info.json
===(call_tool)=== Tool 'search_arxiv' executed successfully.
===(search_arxiv)=== Stored paper information in data\papers_info.json
===(call_tool)=== Tool 'search_arxiv' executed successfully.
===(process_user_query)=== Invoking tool: get_paper_info with args: {'paper_id': '2507.02910v1'}
===(call_tool)=== Tool 'get_paper_info' executed successfully.
===(process_user_query)=== Invoking tool: get_paper_info with args: {'paper_id': '2507.02910v1'}
===(call_tool)=== Tool 'get_paper_info' executed successfully.


========== RESPONSE ==========

I found 2 recent papers on reinforcement learning:

1. "ARLBench: Flexible and Efficient Benchmarking for Hyperpar